# Cosmos Predict 2.5 Inference Demo

This notebook demonstrates how to run inference using the `Inference` class from `cosmos_predict2.inference`.

In [1]:
import os
import sys
from pathlib import Path

# 替换为你系统上 CUDA 12.8 的实际安装路径，通常是 /usr/local/cuda-12.8
cuda_path = "/usr/local/cuda-12.8"

os.environ["CUDA_HOME"] = cuda_path
os.environ["PATH"] = f"{cuda_path}/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{cuda_path}/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# Prefer local checkpoints (avoid HF downloads when files exist)
os.environ.setdefault("COSMOS_CHECKPOINTS_DIR", "/data/LFT-W02_data/junjie/weights")

# Ensure the current directory is in PYTHONPATH so we can import cosmos_predict2
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Set output directory
output_dir = project_root / "outputs" / "video_2_frame_context_00"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Output directory: {output_dir}")

Project root: /data/LFT-W02_data/junjie/cosmos-predict2.5
Output directory: /data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/video_2_frame_context_00


## 1. Setup Arguments

Configure the model setup arguments. You can choose a pre-trained model or specify a local checkpoint.

In [2]:
from cosmos_predict2.config import SetupArguments

# If you want to use a local checkpoint, uncomment and adapt the following:
setup_args = SetupArguments(
    config_file="cosmos_predict2/_src/predict2/configs/video2world/config.py",
    checkpoint_path="/data/LFT-W02_data/junjie/weights/Cosmos-Predict2.5-2B/base/post-trained/81edfebe-bd6a-4039-8c1d-737df1a790bf_ema_bf16.pt",
    context_parallel_size=1,
    output_dir=output_dir,
    model="2B/post-trained",
    keep_going=True,
    disable_guardrails=True,
)

/data/LFT-W02_data/junjie/cosmos-predict2.5/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Initialize Inference Pipeline

Load the model using the `Inference` class.

In [3]:
from cosmos_predict2.inference import Inference

# Initialize the inference pipeline
# Downloads occur only if required weights are missing from COSMOS_CHECKPOINTS_DIR/cache.
pipe = Inference(setup_args)

[01-30 12:38:08|INFO|cosmos_predict2/_src/imaginaire/visualize/video.py:31:<module>] No module named 'ffmpegcv'
[01-30 12:38:21|CRITICAL|cosmos_predict2/_src/predict2/checkpointer/dcp.py:188:<module>] for the back comptiable pytorch! New DefaultLoadPlanner class is created.
[01-30 12:38:21|INFO|cosmos_predict2/_src/predict2/datasets/augmentor_provider.py:94:augmentor_register] registering video_basic_augmentor_v1...
[01-30 12:38:21|INFO|cosmos_predict2/_src/predict2/datasets/augmentor_provider.py:94:augmentor_register] registering video_basic_augmentor_v2...
[01-30 12:38:21|INFO|cosmos_predict2/_src/predict2/datasets/augmentor_provider.py:94:augmentor_register] registering noframedrop_nocameramove_video_augmentor_v1...
[01-30 12:38:21|INFO|cosmos_predict2/_src/predict2/datasets/augmentor_provider.py:94:augmentor_register] registering nocameramove_video_augmentor_v1...
[01-30 12:38:21|INFO|cosmos_predict2/_src/predict2/datasets/augmentor_provider.py:94:augmentor_register] registering im

Config is saved using omegaconf at /data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/video_2_frame_context_00/config.yaml.


[01-30 12:39:01|INFO|cosmos_predict2/inference.py:58:__init__] Saved config to /data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/video_2_frame_context_00/config.yaml


## 3. Run Inference

Define the inference tasks (Text2World or Image2World) and generate videos.

In [6]:
from cosmos_predict2.config import InferenceArguments, InferenceType

# Paths used in run_test.sh / episode_000182.json
input_video_path = Path("/data/LFT-W02_data/junjie/data/test_world_model/episode_000182_30_259.mp4")
prompt_file_path = Path("/data/LFT-W02_data/junjie/data/test_world_model/episode_000182.txt")

# Read prompt content
if prompt_file_path.exists():
    prompt_content = prompt_file_path.read_text().strip()
else:
    prompt_content = "The robot's left hand keep still and don't move. The robot's right hand catch the green apple and put the green apple to the rectangular woven basket."


# Example 1: Text to World
if input_video_path.exists():
    video2world_args = InferenceArguments(
        inference_type=InferenceType.VIDEO2WORLD,
        name="episode_000182_video2world",
        prompt=prompt_content,
        input_path=str(input_video_path),
        num_output_frames=49,
        num_steps=30,
        seed=42,
        guidance=7,
    )

    output_dir = project_root / "outputs" / "episode_000182_30_259"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Run generation for Image2World
    print(f"Generating sample: {video2world_args.name}")
    output_paths_img = pipe.generate([video2world_args], output_dir)
    print(f"Generated video saved at: {output_paths_img}")
else:
    print(f"Input image not found at {input_video_path}, skipping Image2World example.")

Generating sample: episode_000182_video2world
[01-29 17:37:49|INFO|cosmos_predict2/inference.py:84:generate] Generating 1 samples: ['episode_000182_video2world']
[01-29 17:37:49|INFO|cosmos_predict2/inference.py:88:generate] [1/1] Processing sample episode_000182_video2world
[01-29 17:37:49|INFO|cosmos_predict2/inference.py:101:_generate_sample] Saved arguments to /data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/episode_000182_30_259/episode_000182_video2world.json
[01-29 17:37:49|WARNING|cosmos_predict2/inference.py:117:_generate_sample] Guardrail checks on prompt are disabled
[01-29 17:37:49|INFO|cosmos_predict2/inference.py:137:_generate_sample] Generating video with standard mode...
[01-29 17:37:49|INFO|cosmos_predict2/_src/predict2/inference/video2world.py:505:generate_vid2world] Processing video input: /data/LFT-W02_data/junjie/data/test_world_model/episode_000182_30_259.mp4
[01-29 17:37:50|INFO|cosmos_predict2/_src/predict2/inference/video2world.py:182:read_and_process_video]

Generating samples: 100%|██████████| 31/31 [19:28<00:00, 37.71s/it]


[01-29 17:57:47|WARNING|cosmos_predict2/inference.py:174:_generate_sample] Guardrail checks on video are disabled


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[01-29 17:57:51|SUCCESS|cosmos_predict2/inference.py:177:_generate_sample] Saved video to /data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/episode_000182_30_259/episode_000182_video2world.mp4
Generated video saved at: ['/data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/episode_000182_30_259/episode_000182_video2world.mp4']


In [4]:
from cosmos_predict2.config import InferenceArguments, InferenceType

# Paths used in run_test.sh / episode_000182.json
input_video_path = Path("/data/LFT-W02_data/junjie/data/cosmos_test/video_17_frame_context_00_padded_1280x704.png")
prompt_file_path = Path("/data/LFT-W02_data/junjie/data/cosmos_test/video_17_frame_context_00.txt")

# Read prompt content
if prompt_file_path.exists():
    prompt_content = prompt_file_path.read_text().strip()
else:
    prompt_content = "The robot's left hand keep still and don't move. The robot's right hand catch the green apple and put the green apple to the rectangular woven basket."


# Example 1: Text to World
if input_video_path.exists():
    video2world_args = InferenceArguments(
        inference_type=InferenceType.IMAGE2WORLD,
        name="",
        prompt=prompt_content,
        input_path=str(input_video_path),
        num_output_frames=49,
        num_steps=30,
        seed=42,
        guidance=7,
    )
    
    output_dir = project_root / "outputs" / "video_17_frame_context_00" / "video_17_frame_context_00"
    output_dir.mkdir(parents=True, exist_ok=True)
    # Run generation for Image2World
    print(f"Generating sample: {video2world_args.name}")
    output_paths_img = pipe.generate([video2world_args], output_dir)
    print(f"Generated video saved at: {output_paths_img}")
else:
    print(f"Input image not found at {input_video_path}, skipping Image2World example.")

Generating sample: 
[01-30 12:40:21|INFO|cosmos_predict2/inference.py:84:generate] Generating 1 samples: ['']
[01-30 12:40:21|INFO|cosmos_predict2/inference.py:88:generate] [1/1] Processing sample 
[01-30 12:40:21|INFO|cosmos_predict2/inference.py:101:_generate_sample] Saved arguments to /data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/video_17_frame_context_00/video_17_frame_context_00.json
[01-30 12:40:21|WARNING|cosmos_predict2/inference.py:117:_generate_sample] Guardrail checks on prompt are disabled
[01-30 12:40:21|INFO|cosmos_predict2/inference.py:137:_generate_sample] Generating video with standard mode...
[01-30 12:40:21|INFO|cosmos_predict2/_src/predict2/inference/video2world.py:497:generate_vid2world] Processing image input: /data/LFT-W02_data/junjie/data/cosmos_test/video_17_frame_context_00_padded_1280x704.png
[01-30 12:40:22|INFO|cosmos_predict2/_src/predict2/inference/video2world.py:535:generate_vid2world] GPU memory usage after getting data_batch: 19.80 GB
[01-30 12:4

Generating samples: 100%|██████████| 31/31 [19:27<00:00, 37.67s/it]


[01-30 13:00:34|WARNING|cosmos_predict2/inference.py:174:_generate_sample] Guardrail checks on video are disabled


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[01-30 13:00:38|SUCCESS|cosmos_predict2/inference.py:177:_generate_sample] Saved video to /data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/video_17_frame_context_00/video_17_frame_context_00.mp4
Generated video saved at: ['/data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/video_17_frame_context_00/video_17_frame_context_00.mp4']


In [7]:
# Example 2: Image to World (using an asset from the repo if available)
from cosmos_predict2.config import InferenceArguments, InferenceType

input_image_path = Path("/data/LFT-W02_data/junjie/data/test_world_model/robocasa_episode_000001_frames/frame_000001.png")

if input_image_path.exists():
    image2world_args = InferenceArguments(
        inference_type=InferenceType.IMAGE2WORLD,
        name="robocasa_image2world",
        prompt="pick the steak from the cabinet and place it on the counter.",
        input_path=str(input_image_path),
        num_output_frames=49,
        num_steps=30,
        seed=42,
        guidance=7,
    )

    # Run generation for Image2World
    print(f"Generating sample: {image2world_args.name}")
    output_paths_img = pipe.generate([image2world_args], output_dir)
    print(f"Generated video saved at: {output_paths_img}")
else:
    print(f"Input image not found at {input_image_path}, skipping Image2World example.")

Generating sample: robocasa_image2world
[01-29 18:00:30|INFO|cosmos_predict2/inference.py:84:generate] Generating 1 samples: ['robocasa_image2world']
[01-29 18:00:30|INFO|cosmos_predict2/inference.py:88:generate] [1/1] Processing sample robocasa_image2world
[01-29 18:00:30|INFO|cosmos_predict2/inference.py:101:_generate_sample] Saved arguments to /data/LFT-W02_data/junjie/cosmos-predict2.5/outputs/episode_000182_30_259/robocasa_image2world.json
[01-29 18:00:30|WARNING|cosmos_predict2/inference.py:117:_generate_sample] Guardrail checks on prompt are disabled
[01-29 18:00:30|INFO|cosmos_predict2/inference.py:137:_generate_sample] Generating video with standard mode...
[01-29 18:00:30|INFO|cosmos_predict2/_src/predict2/inference/video2world.py:497:generate_vid2world] Processing image input: /data/LFT-W02_data/junjie/data/test_world_model/robocasa_episode_000001_frames/frame_000001.png
[01-29 18:00:32|INFO|cosmos_predict2/_src/predict2/inference/video2world.py:535:generate_vid2world] GPU m

KeyboardInterrupt: 

## 4. Display Results

Display the generated videos directly in the notebook.

In [ ]:
from IPython.display import Video, display


# Display Image2World result if generated
video_path_img = Path(output_paths_img[0])
if video_path_img.exists():
    print("Image2World Result:")
    display(Video(filename=str(video_path_img), embed=True))

Image2World Result:


: 